In [1]:
import sys
from pathlib import Path

# Remonte à la racine du projet depuis notebooks/
project_root = Path.cwd().parent
sys.path.append(str(project_root))
%load_ext autoreload
%autoreload 2

In [2]:
from src.pricing.data import build_pricing_dataset

df = build_pricing_dataset()

print("Nombre de lignes :", len(df))
print("Fréquence globale :", (df["ClaimNb"].sum() / df["Exposure"].sum()))
print("Nombre de sinistres graves :", df["is_large_claim"].sum())
print("Exposure max :", df["Exposure"].max())
print("ClaimNb max :", df["ClaimNb"].max())

Nombre de lignes : 678013
Fréquence globale : 0.10061387819243599
Nombre de sinistres graves : 42
Exposure max : 1.0
ClaimNb max : 4


In [3]:
from src.pricing.features import build_features

df = build_features(df)

print(df[["DrivAge_bucket", "VehAge_bucket", "BM_bucket"]].isna().sum())
print(df[["VehPower_norm", "VehAge_norm", "DrivAge_norm", "BonusMalus_norm"]].describe())

DrivAge_bucket    0
VehAge_bucket     0
BM_bucket         0
dtype: int64
       VehPower_norm    VehAge_norm   DrivAge_norm  BonusMalus_norm
count  678013.000000  678013.000000  678013.000000    678013.000000
mean       -0.553703      -0.859115      -0.329290        -0.891539
std         0.372892       0.113325       0.344816         0.173741
min        -1.000000      -1.000000      -1.000000        -1.000000
25%        -0.818182      -0.960000      -0.609756        -1.000000
50%        -0.636364      -0.880000      -0.365854        -1.000000
75%        -0.454545      -0.780000      -0.097561        -0.844444
max         1.000000       1.000000       1.000000         1.000000


In [4]:
from src.pricing.models import fit_glm_poisson

model_glm = fit_glm_poisson(df)

print("Déviance :", model_glm.deviance)
print("AIC :", model_glm.aic)

Déviance : 212910.34749786658
AIC : 282318.7817089051


In [5]:
from src.pricing.data import train_valid_test_split

train, valid, test = train_valid_test_split(df)

print("Train :", len(train))
print("Valid :", len(valid))
print("Test  :", len(test))
print("Total :", len(train) + len(valid) + len(test), "vs", len(df))

Train : 406807
Valid : 135602
Test  : 135604
Total : 678013 vs 678013


In [6]:
import numpy as np 
model_glm_train = fit_glm_poisson(train)

print("Déviance sur train :", model_glm_train.deviance)

from src.pricing.models import predict_frequency

test_pred = predict_frequency(model_glm_train, test) * test["Exposure"]

y = test["ClaimNb"].values
mu = test_pred.values
mu = np.clip(mu, 1e-10, None)

# Calcul du terme y*log(y/mu) uniquement là où y > 0, pour éviter log(0)
log_term = np.zeros_like(y, dtype=float)
mask = y > 0
log_term[mask] = y[mask] * np.log(y[mask] / mu[mask])

dev_terms = log_term - (y - mu)
deviance_test = 2 * np.sum(dev_terms)

print("Déviance sur test :", deviance_test)

Déviance sur train : 128177.25772677855
Déviance sur test : 43106.69684784079


In [7]:
print("Déviance/obs train :", model_glm_train.deviance / len(train))
print("Déviance/obs test  :", deviance_test / len(test))

Déviance/obs train : 0.3150812491593767
Déviance/obs test  : 0.3178866172667531


In [8]:
from src.pricing.data import get_severity_subset
from src.pricing.models import fit_glm_gamma, predict_severity

train_sev = get_severity_subset(train)
test_sev = get_severity_subset(test)

print("Train sévérité :", len(train_sev))
print("Test sévérité  :", len(test_sev))

model_gamma = fit_glm_gamma(train_sev)
print("\nDéviance Gamma (train) :", model_gamma.deviance)
print("AIC Gamma :", model_gamma.aic)

Train sévérité : 14988
Test sévérité  : 5051

Déviance Gamma (train) : 17897.265498903747
AIC Gamma : 270872.8960406273


In [9]:
from src.pricing.models import predict_severity

test_sev["pred_severity"] = predict_severity(model_gamma, test_sev)

print("Sévérité moyenne observée (test) :", test_sev["ClaimAmount_capped"].mean())
print("Sévérité moyenne prédite (test)  :", test_sev["pred_severity"].mean())
print("Ratio prédit/observé :", test_sev["pred_severity"].mean() / test_sev["ClaimAmount_capped"].mean())

Sévérité moyenne observée (test) : 1882.8932864779251
Sévérité moyenne prédite (test)  : 1794.7183031737827
Ratio prédit/observé : 0.9531704829278564


In [10]:
print("Médiane observée (test) :", test_sev["ClaimAmount_capped"].median())
print("Médiane prédite (test)  :", test_sev["pred_severity"].median())

Médiane observée (test) : 1172.0
Médiane prédite (test)  : 1742.713245422544


In [11]:
test_sev["BM_bucket"] = test_sev["BM_bucket"].astype(str)

comparison = test_sev.groupby("BM_bucket").agg(
    observed_mean=("ClaimAmount_capped", "mean"),
    predicted_mean=("pred_severity", "mean"),
    n=("ClaimAmount_capped", "size")
)
comparison["ratio"] = comparison["predicted_mean"] / comparison["observed_mean"]
print(comparison)

           observed_mean  predicted_mean     n     ratio
BM_bucket                                               
101-125      2006.894529     1993.685411   223  0.993418
126-150      3148.529500     1854.484495    20  0.589000
151+         1346.613333     3630.097232     9  2.695724
50-60        1818.468907     1734.180481  2892  0.953649
61-80        1931.670327     1752.287078  1133  0.907136
81-100       1990.015297     2002.814635   774  1.006432


In [12]:
from src.pricing.models import compute_pure_premium

test["pure_premium"] = compute_pure_premium(model_glm_train, model_gamma, test)

print(test[["pure_premium"]].describe())

        pure_premium
count  135604.000000
mean      216.863019
std       217.257078
min        28.180469
25%       115.089411
50%       148.844047
75%       218.278545
max      8801.588404


In [13]:
total_pure_premium = (test["pure_premium"] * test["Exposure"]).sum()
total_claims_observed = test["ClaimAmount_capped"].sum()  # rappel : sinistres capés à 100k€

print(f"Prime pure totale prédite : {total_pure_premium:,.0f} €")
print(f"Sinistres attritionnels totaux observés : {total_claims_observed:,.0f} €")
print(f"Ratio prédit/observé : {total_pure_premium / total_claims_observed:.4f}")

Prime pure totale prédite : 13,300,272 €
Sinistres attritionnels totaux observés : 10,710,494 €
Ratio prédit/observé : 1.2418


In [14]:
# Combien de sinistres dans le portefeuille test ont un ClaimNb>0 mais ClaimAmount=0 ?
missing_severity_test = test[(test["ClaimNb"] > 0) & (test["ClaimAmount"] == 0)]

print("Nb polices concernées dans test :", len(missing_severity_test))
print("Nb polices avec ClaimNb>0 dans test :", (test["ClaimNb"] > 0).sum())
print("Proportion :", len(missing_severity_test) / (test["ClaimNb"] > 0).sum())

Nb polices concernées dans test : 1889
Nb polices avec ClaimNb>0 dans test : 6952
Proportion : 0.2717203682393556


In [15]:
# On exclut les polices concernées par l'incohérence pour un test de cohérence "propre"
test_clean = test[~((test["ClaimNb"] > 0) & (test["ClaimAmount"] == 0))]

total_pure_premium_clean = (test_clean["pure_premium"] * test_clean["Exposure"]).sum()
total_claims_clean = test_clean["ClaimAmount_capped"].sum()

print(f"Prime pure totale (nettoyée) : {total_pure_premium_clean:,.0f} €")
print(f"Sinistres observés (nettoyé) : {total_claims_clean:,.0f} €")
print(f"Ratio nettoyé : {total_pure_premium_clean / total_claims_clean:.4f}")

Prime pure totale (nettoyée) : 13,025,601 €
Sinistres observés (nettoyé) : 10,710,494 €
Ratio nettoyé : 1.2162


In [16]:
predicted_claims_total = (predict_frequency(model_glm_train, test) * test["Exposure"]).sum()
observed_claims_total = test["ClaimNb"].sum()

print(f"Nombre de sinistres prédit (test) : {predicted_claims_total:,.1f}")
print(f"Nombre de sinistres observé (test) : {observed_claims_total:,.0f}")
print(f"Ratio fréquence prédite/observée : {predicted_claims_total / observed_claims_total:.4f}")

Nombre de sinistres prédit (test) : 7,248.8
Nombre de sinistres observé (test) : 7,368
Ratio fréquence prédite/observée : 0.9838


In [17]:
# Sévérité prédite moyenne sur TOUT test_clean (tous profils, pas seulement les sinistrés)
severity_pred_all = predict_severity(model_gamma, test_clean)
print("Sévérité prédite moyenne (tout le portefeuille clean) :", severity_pred_all.mean())

# Sévérité prédite moyenne sur le sous-échantillon sinistré uniquement (déjà mesuré : 1794.72)
print("Sévérité prédite moyenne (sinistrés uniquement, rappel) :", test_sev["pred_severity"].mean())

# Fréquence prédite moyenne, pondération implicite
freq_pred_all = predict_frequency(model_glm_train, test_clean)
print("Corrélation fréquence prédite / sévérité prédite :", np.corrcoef(freq_pred_all, severity_pred_all)[0,1])

Sévérité prédite moyenne (tout le portefeuille clean) : 1784.8513655268162
Sévérité prédite moyenne (sinistrés uniquement, rappel) : 1794.7183031737827
Corrélation fréquence prédite / sévérité prédite : 0.5192619556380415


In [18]:
# Nombre de sinistres prédit, recalculé spécifiquement sur test_clean
predicted_claims_count_clean = (predict_frequency(model_glm_train, test_clean) * test_clean["Exposure"]).sum()

# Nombre de sinistres réellement payés dans test_clean (ClaimAmount > 0)
observed_claims_count_clean = (test_clean["ClaimAmount"] > 0).sum()

print("Nb sinistres prédit (clean) :", predicted_claims_count_clean)
print("Nb sinistres payés observés (clean) :", observed_claims_count_clean)

# Sévérité moyenne implicite du modèle = prime totale / nb sinistres prédits
implied_severity_predicted = total_pure_premium_clean / predicted_claims_count_clean
# Sévérité moyenne réellement observée par sinistre payé
implied_severity_observed = total_claims_clean / observed_claims_count_clean

print("Sévérité moyenne implicite (modèle) :", implied_severity_predicted)
print("Sévérité moyenne observée (réelle)   :", implied_severity_observed)
print("Ratio sévérité :", implied_severity_predicted / implied_severity_observed)

Nb sinistres prédit (clean) : 7111.448326278257
Nb sinistres payés observés (clean) : 5063
Sévérité moyenne implicite (modèle) : 1831.6382465273446
Sévérité moyenne observée (réelle)   : 2115.444201066561
Ratio sévérité : 0.8658409640887111


In [19]:
# Nombre total d'événements de sinistres (pas de polices) dans test_clean
total_claim_events_clean = test_clean["ClaimNb"].sum()

print("Nb événements de sinistres (ClaimNb, clean) :", total_claim_events_clean)
print("Nb polices payées (clean) :", observed_claims_count_clean)
print("Écart :", total_claim_events_clean - observed_claims_count_clean)

# Combien de polices dans test_clean ont plus d'un sinistre ?
multi_claims = test_clean[test_clean["ClaimNb"] > 1]
print("Polices avec ClaimNb > 1 :", len(multi_claims))
print(multi_claims["ClaimNb"].value_counts())

Nb événements de sinistres (ClaimNb, clean) : 5381
Nb polices payées (clean) : 5063
Écart : 318
Polices avec ClaimNb > 1 : 296
ClaimNb
2    278
3     14
4      4
Name: count, dtype: int64


In [20]:
# Sur le portefeuille TEST COMPLET (pas clean) :
observed_events_total = test["ClaimNb"].sum()          # 7368, déjà connu
events_with_known_severity = test_clean["ClaimNb"].sum()  # 5381, déjà connu
events_with_unknown_severity = observed_events_total - events_with_known_severity

print("Événements à coût connu :", events_with_known_severity)
print("Événements à coût INCONNU :", events_with_unknown_severity)
print("Proportion inconnue :", events_with_unknown_severity / observed_events_total)

# Coût moyen par événement connu
avg_known_severity = total_claims_clean / events_with_known_severity
print("Coût moyen par événement connu :", avg_known_severity)

# Si les événements inconnus avaient le même coût moyen, quel serait le total réel ?
estimated_true_total = total_claims_clean + events_with_unknown_severity * avg_known_severity
print("Estimation du coût total RÉEL (si coût inconnu = coût moyen connu) :", estimated_true_total)

print("\nRatio prédit vs observé PARTIEL (borne basse) :", total_pure_premium / total_claims_observed)
print("Ratio prédit vs coût total ESTIMÉ (borne haute) :", total_pure_premium / estimated_true_total)

Événements à coût connu : 5381
Événements à coût INCONNU : 1987
Proportion inconnue : 0.2696796959826276
Coût moyen par événement connu : 1990.4281713436162
Estimation du coût total RÉEL (si coût inconnu = coût moyen connu) : 14665474.766459763

Ratio prédit vs observé PARTIEL (borne basse) : 1.2417982187390029
Ratio prédit vs coût total ESTIMÉ (borne haute) : 0.9069104526377002


In [21]:
import torch
print(torch.__version__)
print("GPU disponible :", torch.cuda.is_available())

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device utilisé :", device)

2.7.1+cu118
GPU disponible : True
Device utilisé : cuda


In [22]:
import numpy as np

# 1. Prédiction du GLM en log-échelle, sur train et test
# Clamp predictions to avoid log(0) or log(negative)
train_freq_pred = predict_frequency(model_glm_train, train).clip(lower=1e-8)
test_freq_pred = predict_frequency(model_glm_train, test).clip(lower=1e-8)
valid_freq_pred = predict_frequency(model_glm_train, valid).clip(lower=1e-8)

train["glm_log_pred"] = np.log(train_freq_pred)
test["glm_log_pred"] = np.log(test_freq_pred)
valid["glm_log_pred"] = np.log(valid_freq_pred)

print("glm_log_pred stats:")
print(train["glm_log_pred"].describe())
print("Min:", train["glm_log_pred"].min(), "Max:", train["glm_log_pred"].max())
print("Has -inf:", np.isneginf(train["glm_log_pred"]).any())
print("Has inf:", np.isinf(train["glm_log_pred"]).any())
print("Has NaN:", np.isnan(train["glm_log_pred"]).any())

# 2. Les variables continues et catégorielles pour le réseau
continuous_cols = ["VehPower_norm", "VehAge_norm", "DrivAge_norm", "BonusMalus_norm", "Density_log"]
categorical_cols = ["VehBrand_code", "Region_code", "Area_code", "VehGas_code"]

print("\nContinuous columns NaN check:")
print(train[continuous_cols].isna().sum())

print("\nCategorical columns range check:")
categorical_cardinalities = {"VehBrand_code": 11, "Region_code": 21, "Area_code": 6, "VehGas_code": 2}
for col in categorical_cols:
    print(f"{col}: min={train[col].min()}, max={train[col].max()}, has NaN={train[col].isna().any()}, has negative={(train[col] < 0).any()}")
    if train[col].max() >= categorical_cardinalities[col]:
        print(f"  WARNING: max value {train[col].max()} >= cardinality {categorical_cardinalities[col]}")

glm_log_pred stats:
count    406807.000000
mean         -2.332637
std           0.504760
min          -3.609508
25%          -2.666707
50%          -2.467259
75%          -2.115646
max           0.694218
Name: glm_log_pred, dtype: float64
Min: -3.6095076596800104 Max: 0.694218326735503
Has -inf: False
Has inf: False
Has NaN: False

Continuous columns NaN check:
VehPower_norm      0
VehAge_norm        0
DrivAge_norm       0
BonusMalus_norm    0
Density_log        0
dtype: int64

Categorical columns range check:
VehBrand_code: min=0, max=10, has NaN=False, has negative=False
Region_code: min=0, max=20, has NaN=False, has negative=False
Area_code: min=0, max=5, has NaN=False, has negative=False
VehGas_code: min=0, max=1, has NaN=False, has negative=False


In [23]:
from src.pricing.cann import FreMTPL2Dataset

train_dataset = FreMTPL2Dataset(train)
valid_dataset = FreMTPL2Dataset(valid)
test_dataset = FreMTPL2Dataset(test)

print("Taille train dataset :", len(train_dataset))
print("Taille valid dataset :", len(valid_dataset))
print("Taille test dataset :", len(test_dataset))

sample = train_dataset[0]
for key, value in sample.items():
    print(key, "->", value.shape if value.dim() > 0 else value.item(), value.dtype)

Taille train dataset : 406807
Taille valid dataset : 135602
Taille test dataset : 135604
continuous -> torch.Size([5]) torch.float32
categorical -> torch.Size([4]) torch.int64
glm_log_pred -> -2.64168119430542 torch.float32
exposure -> 0.03999999910593033 torch.float32
claim_nb -> 0.0 torch.float32


In [24]:
from src.pricing.cann import CANNFrequencyNet, CATEGORICAL_CARDINALITIES

model = CANNFrequencyNet(
    n_continuous=5,
    categorical_cardinalities=CATEGORICAL_CARDINALITIES,
)

sample = train_dataset[0]
with torch.no_grad():
    log_lambda_cann = model(
        sample["continuous"].unsqueeze(0),
        sample["categorical"].unsqueeze(0),
        sample["glm_log_pred"].unsqueeze(0),
    )

print("log(lambda) GLM  :", sample["glm_log_pred"].item())
print("log(lambda) CANN :", log_lambda_cann.item())
print("Écart :", abs(sample["glm_log_pred"].item() - log_lambda_cann.item()))

log(lambda) GLM  : -2.64168119430542
log(lambda) CANN : -2.64168119430542
Écart : 0.0


In [25]:
"""
from src.pricing.cann import CANNFrequencyNet, CATEGORICAL_CARDINALITIES, train_cann

# Modèle neuf, avec les poids ré-initialisés à zéro sur la dernière couche
model = CANNFrequencyNet(
    n_continuous=5,
    categorical_cardinalities=CATEGORICAL_CARDINALITIES,
)

model_cann, history = train_cann(model, train_dataset, valid_dataset, n_epochs=2, lr=1e-3, device=device)"""

'\nfrom src.pricing.cann import CANNFrequencyNet, CATEGORICAL_CARDINALITIES, train_cann\n\n# Modèle neuf, avec les poids ré-initialisés à zéro sur la dernière couche\nmodel = CANNFrequencyNet(\n    n_continuous=5,\n    categorical_cardinalities=CATEGORICAL_CARDINALITIES,\n)\n\nmodel_cann, history = train_cann(model, train_dataset, valid_dataset, n_epochs=2, lr=1e-3, device=device)'

In [26]:
glm_valid_pred = predict_frequency(model_glm_train, valid) * valid["Exposure"]

y = valid["ClaimNb"].values
mu = glm_valid_pred.values
mu = np.clip(mu, 1e-10, None)

log_term = np.zeros_like(y, dtype=float)
mask = y > 0
log_term[mask] = y[mask] * np.log(y[mask] / mu[mask])

dev_terms = log_term - (y - mu)
deviance_glm_valid = 2 * np.sum(dev_terms)

print("Déviance GLM sur valid (brute) :", deviance_glm_valid)
print("Déviance GLM sur valid (par obs) :", deviance_glm_valid / len(valid))

Déviance GLM sur valid (brute) : 41654.466029930576
Déviance GLM sur valid (par obs) : 0.3071817969493855


In [27]:
for name, p in model.named_parameters():
    print(name, p.requires_grad, p.shape)

embeddings.VehBrand_code.weight True torch.Size([11, 4])
embeddings.Region_code.weight True torch.Size([21, 4])
embeddings.Area_code.weight True torch.Size([6, 4])
embeddings.VehGas_code.weight True torch.Size([2, 4])
mlp.0.weight True torch.Size([32, 21])
mlp.0.bias True torch.Size([32])
mlp.2.weight True torch.Size([32, 32])
mlp.2.bias True torch.Size([32])
mlp.4.weight True torch.Size([1, 32])
mlp.4.bias True torch.Size([1])


In [28]:
total_norm = sum(p.grad.norm().item()**2 for p in model.parameters() if p.grad is not None) ** 0.5
print(total_norm)

0.0


In [29]:
for name, p in model.named_parameters():
    if p.grad is not None:
        print(f"{name}: grad_norm = {p.grad.norm().item():.6f}")

In [30]:
from torch.utils.data import DataLoader

model_cann.eval()
with torch.no_grad():
    batch = next(iter(DataLoader(valid_dataset, batch_size=4096, shuffle=False)))
    continuous = batch["continuous"].to(device)
    categorical = batch["categorical"].to(device)
    glm_log_pred = batch["glm_log_pred"].to(device)

    cann_output = model_cann(continuous, categorical, glm_log_pred)
    residual = cann_output - glm_log_pred

    print("Résidu appris — moyenne :", residual.mean().item())
    print("Résidu appris — std     :", residual.std().item())
    print("Résidu appris — min/max :", residual.min().item(), residual.max().item())

NameError: name 'model_cann' is not defined

In [31]:
# mu_GLM = exposure * lambda_GLM -- c'est le "working weight" du papier (éq 3.6)
train["log_mu_glm"] = train["glm_log_pred"] + np.log(train["Exposure"])
valid["log_mu_glm"] = valid["glm_log_pred"] + np.log(valid["Exposure"])

print(train["log_mu_glm"].describe())

count    406807.000000
mean         -3.410873
std           1.167289
min          -9.169606
25%          -4.045326
50%          -3.069622
75%          -2.602055
max           0.282785
Name: log_mu_glm, dtype: float64


In [45]:

from src.pricing.cann import PairInteractionNet, PairDataset, poisson_deviance_loss_v2
from torch.utils.data import DataLoader

train_pair = PairDataset(train, "DrivAge_norm", "BonusMalus_norm")
valid_pair = PairDataset(valid, "DrivAge_norm", "BonusMalus_norm")

model_pair = PairInteractionNet().to(device)
optimizer = torch.optim.Adam(model_pair.parameters(), lr=1e-3)

train_loader = DataLoader(train_pair, batch_size=4096, shuffle=True)
valid_loader = DataLoader(valid_pair, batch_size=4096, shuffle=False)

best_valid = float("inf")
for epoch in range(2):
    model_pair.train()
    for batch in train_loader:
        var1 = batch["var1"].to(device)
        var2 = batch["var2"].to(device)
        log_mu_glm = batch["log_mu_glm"].to(device)
        claim_nb = batch["claim_nb"].to(device)

        optimizer.zero_grad()
        log_lambda = model_pair(var1, var2, log_mu_glm)
        loss = poisson_deviance_loss_v2(log_lambda, claim_nb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_pair.parameters(), max_norm=1.0)
        optimizer.step()

    model_pair.eval()
    valid_losses = []
    with torch.no_grad():
        for batch in valid_loader:
            var1 = batch["var1"].to(device)
            var2 = batch["var2"].to(device)
            log_mu_glm = batch["log_mu_glm"].to(device)
            claim_nb = batch["claim_nb"].to(device)

            log_lambda = model_pair(var1, var2, log_mu_glm)
            loss = poisson_deviance_loss_v2(log_lambda, claim_nb)
            valid_losses.append(loss.item())

    avg_valid = sum(valid_losses) / len(valid_losses)
    if avg_valid < best_valid:
        best_valid = avg_valid

    if epoch % 5 == 0 or epoch == 49:
        print(f"Epoch {epoch+1}/50 — valid_loss: {avg_valid:.4f}")

print(f"\nMeilleur valid_loss (DrivAge x BonusMalus) : {best_valid:.4f}")
print(f"Référence GLM sur valid : 0.3072")

Epoch 1/50 — valid_loss: 0.3096

Meilleur valid_loss (DrivAge x BonusMalus) : 0.3095
Référence GLM sur valid : 0.3072


In [46]:
from src.pricing.cann import PairInteractionNet, PairDataset, poisson_deviance_loss_v2
from torch.utils.data import DataLoader
# 1. Est-ce que le gradient circule ?
batch = next(iter(train_loader))
var1 = batch["var1"].to(device)
var2 = batch["var2"].to(device)
log_mu_glm = batch["log_mu_glm"].to(device)
claim_nb = batch["claim_nb"].to(device)

log_lambda = model_pair(var1, var2, log_mu_glm)
loss = poisson_deviance_loss_v2(log_lambda, claim_nb)
loss.backward()

for name, p in model_pair.named_parameters():
    if p.grad is not None:
        print(f"{name}: grad_norm = {p.grad.norm().item():.6f}")

# 2. Le résidu appris a-t-il une vraie amplitude après 50 epochs ?
model_pair.eval()
with torch.no_grad():
    batch = next(iter(valid_loader))
    var1 = batch["var1"].to(device)
    var2 = batch["var2"].to(device)
    log_mu_glm = batch["log_mu_glm"].to(device)

    log_lambda = model_pair(var1, var2, log_mu_glm)
    residual = log_lambda - log_mu_glm

    print("Résidu — moyenne :", residual.mean().item())
    print("Résidu — std     :", residual.std().item())
    print("Résidu — min/max :", residual.min().item(), residual.max().item())

mlp.0.weight: grad_norm = 0.000121
mlp.0.bias: grad_norm = 0.000051
mlp.2.weight: grad_norm = 0.000279
mlp.2.bias: grad_norm = 0.000068
mlp.4.weight: grad_norm = 0.005261
mlp.4.bias: grad_norm = 0.002687
Résidu — moyenne : -0.005643647629767656
Résidu — std     : 0.002313283970579505
Résidu — min/max : -0.0078084468841552734 0.0070955753326416016


In [47]:
"""
train_pair = PairDataset(train, "DrivAge_norm", "BonusMalus_norm")
valid_pair = PairDataset(valid, "DrivAge_norm", "BonusMalus_norm")

model_pair = PairInteractionNet().to(device)
optimizer = torch.optim.Adam(model_pair.parameters(), lr=3e-3)

train_loader = DataLoader(train_pair, batch_size=4096, shuffle=True)
valid_loader = DataLoader(valid_pair, batch_size=4096, shuffle=False)

best_valid = float("inf")
best_epoch = 0
for epoch in range(150):
    model_pair.train()
    for batch in train_loader:
        var1 = batch["var1"].to(device)
        var2 = batch["var2"].to(device)
        log_mu_glm = batch["log_mu_glm"].to(device)
        claim_nb = batch["claim_nb"].to(device)

        optimizer.zero_grad()
        log_lambda = model_pair(var1, var2, log_mu_glm)
        loss = poisson_deviance_loss_v2(log_lambda, claim_nb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_pair.parameters(), max_norm=1.0)
        optimizer.step()

    model_pair.eval()
    valid_losses = []
    with torch.no_grad():
        for batch in valid_loader:
            var1 = batch["var1"].to(device)
            var2 = batch["var2"].to(device)
            log_mu_glm = batch["log_mu_glm"].to(device)
            claim_nb = batch["claim_nb"].to(device)

            log_lambda = model_pair(var1, var2, log_mu_glm)
            loss = poisson_deviance_loss_v2(log_lambda, claim_nb)
            valid_losses.append(loss.item())

    avg_valid = sum(valid_losses) / len(valid_losses)
    if avg_valid < best_valid:
        best_valid = avg_valid
        best_epoch = epoch + 1

    if epoch % 10 == 0 or epoch == 149:
        print(f"Epoch {epoch+1}/150 — valid_loss: {avg_valid:.4f}")

print(f"\nMeilleur valid_loss : {best_valid:.4f} (epoch {best_epoch})")
print(f"Référence GLM sur valid : 0.3072")
"""

'\ntrain_pair = PairDataset(train, "DrivAge_norm", "BonusMalus_norm")\nvalid_pair = PairDataset(valid, "DrivAge_norm", "BonusMalus_norm")\n\nmodel_pair = PairInteractionNet().to(device)\noptimizer = torch.optim.Adam(model_pair.parameters(), lr=3e-3)\n\ntrain_loader = DataLoader(train_pair, batch_size=4096, shuffle=True)\nvalid_loader = DataLoader(valid_pair, batch_size=4096, shuffle=False)\n\nbest_valid = float("inf")\nbest_epoch = 0\nfor epoch in range(150):\n    model_pair.train()\n    for batch in train_loader:\n        var1 = batch["var1"].to(device)\n        var2 = batch["var2"].to(device)\n        log_mu_glm = batch["log_mu_glm"].to(device)\n        claim_nb = batch["claim_nb"].to(device)\n\n        optimizer.zero_grad()\n        log_lambda = model_pair(var1, var2, log_mu_glm)\n        loss = poisson_deviance_loss_v2(log_lambda, claim_nb)\n        loss.backward()\n        torch.nn.utils.clip_grad_norm_(model_pair.parameters(), max_norm=1.0)\n        optimizer.step()\n\n    model_

In [48]:
"""
from src.pricing.cann import GroupInteractionNet, GroupDataset

group_continuous_cols = ["VehPower_norm", "VehAge_norm", "VehGas_code"]

train_group = GroupDataset(train, group_continuous_cols, "VehBrand_code")
valid_group = GroupDataset(valid, group_continuous_cols, "VehBrand_code")

model_group = GroupInteractionNet(n_continuous=3, brand_cardinality=11).to(device)
optimizer = torch.optim.Adam(model_group.parameters(), lr=3e-3)

train_loader = DataLoader(train_group, batch_size=4096, shuffle=True)
valid_loader = DataLoader(valid_group, batch_size=4096, shuffle=False)

best_valid = float("inf")
best_epoch = 0
for epoch in range(150):
    model_group.train()
    for batch in train_loader:
        continuous = batch["continuous"].to(device)
        brand_code = batch["brand_code"].to(device)
        log_mu_glm = batch["log_mu_glm"].to(device)
        claim_nb = batch["claim_nb"].to(device)

        optimizer.zero_grad()
        log_lambda = model_group(continuous, brand_code, log_mu_glm)
        loss = poisson_deviance_loss_v2(log_lambda, claim_nb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_group.parameters(), max_norm=1.0)
        optimizer.step()

    model_group.eval()
    valid_losses = []
    with torch.no_grad():
        for batch in valid_loader:
            continuous = batch["continuous"].to(device)
            brand_code = batch["brand_code"].to(device)
            log_mu_glm = batch["log_mu_glm"].to(device)
            claim_nb = batch["claim_nb"].to(device)

            log_lambda = model_group(continuous, brand_code, log_mu_glm)
            loss = poisson_deviance_loss_v2(log_lambda, claim_nb)
            valid_losses.append(loss.item())

    avg_valid = sum(valid_losses) / len(valid_losses)
    if avg_valid < best_valid:
        best_valid = avg_valid
        best_epoch = epoch + 1

    if epoch % 10 == 0 or epoch == 149:
        print(f"Epoch {epoch+1}/150 — valid_loss: {avg_valid:.4f}")

print(f"\nMeilleur valid_loss : {best_valid:.4f} (epoch {best_epoch})")
print(f"Référence GLM sur valid : 0.3072")
"""

'\nfrom src.pricing.cann import GroupInteractionNet, GroupDataset\n\ngroup_continuous_cols = ["VehPower_norm", "VehAge_norm", "VehGas_code"]\n\ntrain_group = GroupDataset(train, group_continuous_cols, "VehBrand_code")\nvalid_group = GroupDataset(valid, group_continuous_cols, "VehBrand_code")\n\nmodel_group = GroupInteractionNet(n_continuous=3, brand_cardinality=11).to(device)\noptimizer = torch.optim.Adam(model_group.parameters(), lr=3e-3)\n\ntrain_loader = DataLoader(train_group, batch_size=4096, shuffle=True)\nvalid_loader = DataLoader(valid_group, batch_size=4096, shuffle=False)\n\nbest_valid = float("inf")\nbest_epoch = 0\nfor epoch in range(150):\n    model_group.train()\n    for batch in train_loader:\n        continuous = batch["continuous"].to(device)\n        brand_code = batch["brand_code"].to(device)\n        log_mu_glm = batch["log_mu_glm"].to(device)\n        claim_nb = batch["claim_nb"].to(device)\n\n        optimizer.zero_grad()\n        log_lambda = model_group(continuo

In [50]:
from src.pricing.cann import GroupInteractionNet, GroupDataset

group_continuous_cols = ["VehPower_norm", "VehAge_norm", "VehGas_code"]

train_group = GroupDataset(train, group_continuous_cols, "VehBrand_code")
valid_group = GroupDataset(valid, group_continuous_cols, "VehBrand_code")

model_group = GroupInteractionNet(n_continuous=3, brand_cardinality=11).to(device)
optimizer = torch.optim.Adam(model_group.parameters(), lr=3e-3)

train_loader = DataLoader(train_group, batch_size=4096, shuffle=True)
valid_loader = DataLoader(valid_group, batch_size=4096, shuffle=False)

best_valid = float("inf")
best_epoch = 0
for epoch in range(400):
    model_group.train()
    for batch in train_loader:
        continuous = batch["continuous"].to(device)
        brand_code = batch["brand_code"].to(device)
        log_mu_glm = batch["log_mu_glm"].to(device)
        claim_nb = batch["claim_nb"].to(device)

        optimizer.zero_grad()
        log_lambda = model_group(continuous, brand_code, log_mu_glm)
        loss = poisson_deviance_loss_v2(log_lambda, claim_nb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_group.parameters(), max_norm=1.0)
        optimizer.step()

    model_group.eval()
    valid_losses = []
    with torch.no_grad():
        for batch in valid_loader:
            continuous = batch["continuous"].to(device)
            brand_code = batch["brand_code"].to(device)
            log_mu_glm = batch["log_mu_glm"].to(device)
            claim_nb = batch["claim_nb"].to(device)

            log_lambda = model_group(continuous, brand_code, log_mu_glm)
            loss = poisson_deviance_loss_v2(log_lambda, claim_nb)
            valid_losses.append(loss.item())

    avg_valid = sum(valid_losses) / len(valid_losses)
    if avg_valid < best_valid:
        best_valid = avg_valid
        best_epoch = epoch + 1

    if epoch % 10 == 0 or epoch == 399:
        print(f"Epoch {epoch+1}/400 — valid_loss: {avg_valid:.4f}")

print(f"\nMeilleur valid_loss : {best_valid:.4f} (epoch {best_epoch})")
print(f"Référence GLM sur valid : 0.3072")

Epoch 1/400 — valid_loss: 0.3096


KeyboardInterrupt: 

In [51]:
from src.pricing.cann import train_group_interaction

model_group = GroupInteractionNet(n_continuous=3, brand_cardinality=11).to(device)
optimizer = torch.optim.Adam(model_group.parameters(), lr=3e-3)

model_group, best_valid, best_epoch = train_group_interaction(
    model_group, train_loader, valid_loader, n_epochs=400, optimizer=optimizer, device=device
)

Epoch 1/400 — valid_loss: 0.3095


KeyboardInterrupt: 

In [37]:
# D'abord, calculer log_mu_glm pour test (comme on l'a fait pour train/valid)
test["log_mu_glm"] = test["glm_log_pred"] + np.log(test["Exposure"])

test_group = GroupDataset(test, group_continuous_cols, "VehBrand_code")
test_loader = DataLoader(test_group, batch_size=4096, shuffle=False)

model_group.eval()
test_losses = []
with torch.no_grad():
    for batch in test_loader:
        continuous = batch["continuous"].to(device)
        brand_code = batch["brand_code"].to(device)
        log_mu_glm = batch["log_mu_glm"].to(device)
        claim_nb = batch["claim_nb"].to(device)

        log_lambda = model_group(continuous, brand_code, log_mu_glm)
        loss = poisson_deviance_loss_v2(log_lambda, claim_nb)
        test_losses.append(loss.item())

test_loss_group = sum(test_losses) / len(test_losses)

print("Loss du modèle d'interaction sur test :", test_loss_group)
print("Référence GLM sur test : 0.31788661726675316")  # valeur qu'on avait calculée bien plus tôt

NameError: name 'poisson_deviance_loss_v2' is not defined

In [38]:
from src.pricing.models import fit_ngboost_severity, predict_ngboost_severity

model_ngboost = fit_ngboost_severity(train_sev, n_estimators=300)

ngboost_preds = predict_ngboost_severity(model_ngboost, test_sev)

print("Sévérité moyenne observée (test) :", test_sev["ClaimAmount_capped"].mean())
print("Sévérité moyenne prédite NGBoost  :", ngboost_preds["pred_mean"].mean())
print("Ratio prédit/observé NGBoost :", ngboost_preds["pred_mean"].mean() / test_sev["ClaimAmount_capped"].mean())

# Comparaison directe au GLM Gamma (rappel : ratio 0.953)
print("\nRappel — Ratio GLM Gamma : 0.9532")

# Test de couverture de l'intervalle 90% -- LE test important pour un modèle distributionnel
covered = (
    (test_sev["ClaimAmount_capped"].values >= ngboost_preds["pred_lower_90"].values) &
    (test_sev["ClaimAmount_capped"].values <= ngboost_preds["pred_upper_90"].values)
)
print(f"\nCouverture empirique de l'intervalle 90% : {covered.mean():.2%}")
print("Couverture théorique visée : 90%")

[iter 0] loss=8.4915 val_loss=0.0000 scale=0.5000 norm=0.9816
[iter 50] loss=8.4765 val_loss=0.0000 scale=1.0000 norm=1.9295
[iter 100] loss=8.4688 val_loss=0.0000 scale=0.5000 norm=0.9606
[iter 150] loss=8.4624 val_loss=0.0000 scale=1.0000 norm=1.9166
[iter 200] loss=8.4571 val_loss=0.0000 scale=0.2500 norm=0.4784
[iter 250] loss=8.4513 val_loss=0.0000 scale=1.0000 norm=1.9068
Sévérité moyenne observée (test) : 1882.8932864779251
Sévérité moyenne prédite NGBoost  : 1781.6563019061573
Ratio prédit/observé NGBoost : 0.9462332861353294

Rappel — Ratio GLM Gamma : 0.9532

Couverture empirique de l'intervalle 90% : 90.58%
Couverture théorique visée : 90%


In [39]:
# Sur les polices sinistrées uniquement (test_sev), on a déjà pred_severity (GLM) 
# et on peut calculer la fréquence prédite correspondante
test_sev_freq = predict_frequency(model_glm_train, test_sev)

# Résidus : sinistre réel vs prédiction, pour chaque dimension
severity_residual = test_sev["ClaimAmount_capped"] - test_sev["pred_severity"]
freq_residual = test_sev["ClaimNb"] - test_sev_freq  # rarement utile seul, mais on regarde la corrélation brute

from scipy.stats import spearmanr

corr, pval = spearmanr(test_sev_freq, test_sev["ClaimAmount_capped"])
print(f"Corrélation de Spearman (fréquence prédite vs sévérité observée) : {corr:.4f}")
print(f"P-value : {pval:.4f}")

Corrélation de Spearman (fréquence prédite vs sévérité observée) : 0.0784
P-value : 0.0000


In [ ]:
import joblib
from pathlib import Path

Path("../models").mkdir(exist_ok=True)

# GLM (rapide à réentraîner, mais autant le sauvegarder aussi)
joblib.dump(model_glm_train, "../models/glm_poisson.pkl")
joblib.dump(model_gamma, "../models/glm_gamma.pkl")

# NGBoost
joblib.dump(model_ngboost, "../models/ngboost_severity.pkl")

# CANN groupe (le modèle final, 400 epochs)
torch.save(model_group.state_dict(), "../models/cann_group_interaction.pt")

In [40]:
import shap
import numpy as np
import torch
from src.pricing.cann import get_shap_input_matrix, ResidualMLPWrapper

continuous_cols_group = ["VehPower_norm", "VehAge_norm", "VehGas_code"]

# Matrice d'entrée : continues + embedding VehBrand (dim 2)
X_test_shap = get_shap_input_matrix(model_group, test, continuous_cols_group, "VehBrand_code", device)

# Échantillon de fond pour l'explainer (100 points suffisent, SHAP est coûteux)
background_idx = np.random.choice(len(X_test_shap), 100, replace=False)
background = torch.tensor(X_test_shap[background_idx], dtype=torch.float32).to(device)

wrapper = ResidualMLPWrapper(model_group).to(device)
wrapper.eval()

explainer = shap.DeepExplainer(wrapper, background)

# Explication sur un sous-échantillon de test (limiter à 500 pour la vitesse)
sample_idx = np.random.choice(len(X_test_shap), 500, replace=False)
X_sample = torch.tensor(X_test_shap[sample_idx], dtype=torch.float32).to(device)

shap_values = explainer.shap_values(X_sample)

feature_names = continuous_cols_group + ["VehBrand_emb_0", "VehBrand_emb_1"]
print("Shape des valeurs SHAP :", np.array(shap_values).shape)

c:\Users\marie\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Shape des valeurs SHAP : (500, 5, 1)


In [42]:
import numpy as np
import pandas as pd

shap_array = np.array(shap_values).squeeze()

mean_abs_shap = np.abs(shap_array).mean(axis=0)
importance_df = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False)

print(importance_df)

# Agrégation des 2 dimensions d'embedding VehBrand en une seule importance
vehbrand_importance = mean_abs_shap[3] + mean_abs_shap[4]
print(f"\nImportance agrégée VehBrand : {vehbrand_importance:.6f}")

          feature  mean_abs_shap
0   VehPower_norm            0.0
1     VehAge_norm            0.0
2     VehGas_code            0.0
3  VehBrand_emb_0            0.0
4  VehBrand_emb_1            0.0

Importance agrégée VehBrand : 0.000000
